# UD4.01 — Matplotlib: arquitectura y gráficos básicos

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
UD4 — Visualización de datos · 14 horas

Criterio 1.d · Material de partida de la práctica P4.1

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

- **Explicar la arquitectura de Matplotlib** y la jerarquía Figure / Axes / Axis
- **Usar las dos interfaces** (pyplot y orientada a objetos) y justificar cuál corresponde
- **Elegir el tipo de gráfico** adecuado al tipo de dato y a la pregunta que se responde
- **Construir figuras con varios paneles** con `subplots` y con `GridSpec`
- **Personalizar** títulos, etiquetas, leyendas, colores y anotaciones
- **Exportar figuras** en el formato correcto según el destino, sabiendo lo que pesa cada uno
- **Reconocer las decisiones de diseño** que hacen que un gráfico engañe

## 1. Introducción a Matplotlib

### ¿Qué es Matplotlib?

Matplotlib es la biblioteca de visualización sobre la que se apoya todo lo demás en
Python. La escribió John Hunter en 2003 imitando la interfaz de MATLAB, y hoy es la
capa de dibujo que usan por debajo Pandas (`df.plot()`), Seaborn y `scikit-learn`.

### ¿Por qué Matplotlib, si hay opciones más cómodas?

- **Es la base del ecosistema.** Cuando una biblioteca de más alto nivel no hace
  exactamente lo que necesitas, lo que te queda debajo es Matplotlib. Saber tocar el
  objeto `Axes` que ha creado Seaborn es lo que permite arreglar el gráfico en lugar
  de rendirse.
- **Control total.** Cualquier elemento del gráfico es un objeto que se puede
  modificar.
- **Salida vectorial.** PDF y SVG con calidad de imprenta, y ficheros pequeños.
- **Funciona sin navegador.** Genera imágenes en un servidor sin entorno gráfico, que
  es donde se ejecutan los informes automáticos.

Lo que **no** aporta es comodidad: hace falta escribir bastante para conseguir un
gráfico presentable. De eso se ocupan Seaborn (cuadernos 02 y 03) y Plotly (cuaderno
04), y las tres bibliotecas se comparan con números medidos en la práctica P4.1.

In [ ]:
import io
import os
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# El estado del generador aleatorio se fija una sola vez y aquí. Todo lo que se dibuja
# en este cuaderno tiene que salir igual en tu ordenador y en el de al lado: si no, no
# se puede discutir un gráfico en clase.
rng = np.random.default_rng(20262027)

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("matplotlib", matplotlib.__version__)
print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)

## 2. Arquitectura: Figure, Axes y Axis

Esta es la parte que más se salta y la que explica el 90 % de los errores. Matplotlib
tiene tres niveles, y los nombres se parecen demasiado:

```
Figure  ── el lienzo entero. Lo que se guarda en un fichero.
  └── Axes ── UNA zona de dibujo con sus ejes. Es donde se pinta.
        ├── Axis ── UN eje: el X, o el Y. Controla límites, marcas y escala.
        ├── Title, Legend
        └── Los datos dibujados (Line2D, PathCollection, Rectangle...)
```

| Objeto | Es | Ojo con |
|---|---|---|
| `Figure` | El lienzo. Tiene tamaño en pulgadas y resolución en puntos por pulgada | Una figura puede contener muchos `Axes` |
| `Axes` | Una zona de dibujo. **Es el objeto con el que se trabaja** | El nombre está en plural pero **es uno solo** |
| `Axis` | Un eje concreto, el X o el Y | `ax.xaxis` y `ax.yaxis` son objetos `Axis` |

La confusión entre `Axes` (zona de dibujo) y `Axis` (un eje) viene del inglés y no
tiene arreglo: hay que memorizarla. La regla práctica es que **casi todo lo que vas a
escribir son métodos de un `Axes`**, y que se llaman `ax.set_algo(...)`.

En lugar de describirlo, vamos a dibujar la anatomía de una figura con la propia
Matplotlib, señalando cada pieza con su nombre.

In [ ]:
# La anatomía de una figura, dibujada con Matplotlib.
fig = plt.figure(figsize=(11, 6), facecolor="#f4f4f4")
ax = fig.add_axes((0.18, 0.16, 0.62, 0.66))  # (izquierda, abajo, ancho, alto) en 0-1

x = np.linspace(0, 10, 40)
ax.plot(x, np.sin(x), marker="o", markersize=4, label="sin(x)")
ax.plot(x, np.cos(x), linestyle="--", label="cos(x)")
ax.set_title("Title: el título de este Axes")
ax.set_xlabel("XLabel: la etiqueta del eje X")
ax.set_ylabel("YLabel: la etiqueta del eje Y")
ax.legend(loc="upper right", title="Legend")
ax.grid(True, alpha=0.3)

# Rotulamos las piezas usando coordenadas de la figura (0-1 en todo el lienzo), que es
# lo que permite escribir fuera del area de dibujo.
anotaciones = [
    (0.03, 0.94, "Figure: el lienzo entero,\nlo que se guarda al fichero", "left"),
    (0.49, 0.05, "Axis X (ax.xaxis):\nlímites, marcas y escala", "center"),
    (0.03, 0.49, "Axis Y\n(ax.yaxis)", "left"),
    (0.83, 0.50, "Axes: la zona de dibujo.\nEs el objeto con el que\nse trabaja", "left"),
]
for fx, fy, texto, alineacion in anotaciones:
    fig.text(fx, fy, texto, ha=alineacion, va="center", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="#888888"))

fig.suptitle("Anatomía de una figura de Matplotlib", fontsize=14, fontweight="bold")
plt.show()

print("El objeto figura :", type(fig).__name__)
print("El objeto axes   :", type(ax).__name__)
print("Los objetos axis :", type(ax.xaxis).__name__, "y", type(ax.yaxis).__name__)
print()
print("Axes que contiene la figura:", len(fig.axes))

## 3. Dos interfaces: pyplot y orientada a objetos

Matplotlib se puede usar de dos formas, y conviene saber distinguirlas porque los
ejemplos que hay por internet mezclan las dos sin avisar.

### 3.1 La interfaz `pyplot`

`plt.plot()`, `plt.title()`, `plt.xlabel()`... son funciones que actúan sobre **la
figura actual**, una variable global que Matplotlib mantiene por dentro. Es la
herencia de MATLAB.

- **A favor:** para un gráfico rápido se escribe menos.
- **En contra:** «la figura actual» es estado global. En cuanto hay dos figuras, o el
  gráfico se hace dentro de una función, deja de estar claro sobre qué se está
  dibujando. Es una fuente de errores difíciles de leer.

In [ ]:
# Interfaz pyplot: corta, y con estado global por debajo.
x = np.linspace(0, 2 * np.pi, 100)

plt.figure(figsize=(10, 4))
plt.plot(x, np.sin(x), label="sin(x)")   # dibuja en "la figura actual"
plt.plot(x, np.cos(x), label="cos(x)")
plt.title("Funciones trigonométricas — interfaz pyplot")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("La figura sobre la que se ha dibujado no está en ninguna variable.")
print("Matplotlib la guardaba por dentro, y `plt.show()` la ha cerrado.")

### 3.2 La interfaz orientada a objetos, que es la que hay que usar

`fig, ax = plt.subplots()` devuelve los dos objetos y a partir de ahí todo es
explícito: `ax.plot(...)`, `ax.set_title(...)`. Dos líneas más, y a cambio:

- Se sabe siempre sobre qué se dibuja.
- Una función puede recibir un `ax` y dibujar dentro, que es lo que permite componer
  figuras grandes con piezas pequeñas.
- Es la interfaz que usan Seaborn (`ax=`) y `df.plot(ax=)`.

**Norma del módulo:** en las prácticas, todo gráfico se hace con la interfaz
orientada a objetos. `plt.` solo se admite para `plt.subplots()`, `plt.show()` y
`plt.close()`.

In [ ]:
# La misma figura con la interfaz orientada a objetos.
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)")
ax.set_title("Funciones trigonométricas — interfaz orientada a objetos")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("Ahora la figura y la zona de dibujo están en variables:")
print("  fig ->", fig)
print("  ax  ->", ax)

### 3.3 Por qué importa: una función que dibuja

Esta es la razón de fondo, y no es estética. Una función que recibe un `ax` es
**reutilizable**: sirve para un gráfico suelto y para un panel de una figura de nueve.
Con `plt.` esto no se puede escribir.

Es la misma idea estructural que en la UD2 se aplicó a `servicios/`: la función
**calcula y dibuja, pero no decide dónde ni imprime nada**. Ni `plt.show()`, ni
`savefig`, ni `print`. Eso lo decide quien la llama.

In [ ]:
def dibuja_serie(ax, x, y, titulo, color=None):
    """Dibuja una serie con su media en un Axes que se recibe de fuera.

    No crea la figura, no la muestra y no la guarda: solo dibuja donde le digan.
    Devuelve el Axes para poder encadenar llamadas.
    """
    ax.plot(x, y, color=color, linewidth=1.8)
    ax.axhline(y.mean(), color="0.4", linestyle=":", linewidth=1.2,
               label=f"media = {y.mean():.2f}")
    ax.set_title(titulo, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    return ax


series = {
    "Ruido blanco": rng.normal(0, 1, 200),
    "Paseo aleatorio": np.cumsum(rng.normal(0, 1, 200)),
    "Tendencia + ruido": np.linspace(0, 5, 200) + rng.normal(0, 0.6, 200),
}

# La misma función, usada dos veces con destinos distintos: primero suelta...
fig, ax = plt.subplots(figsize=(9, 3))
dibuja_serie(ax, np.arange(200), series["Paseo aleatorio"], "Una sola serie")
plt.show()

# ...y después como pieza de una figura de tres paneles.
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (nombre, valores) in zip(axes, series.items()):
    dibuja_serie(ax, np.arange(200), valores, nombre)
fig.suptitle("La misma función, tres veces", fontweight="bold")
fig.tight_layout()
plt.show()

## 4. Los cuatro gráficos básicos

Antes de la sintaxis, la decisión: **cada tipo de gráfico responde a un tipo de
pregunta**. Elegir mal el tipo no es un problema de estética, es un problema de que el
gráfico no contesta lo que se le pregunta.

| Pregunta | Gráfico | Método |
|---|---|---|
| ¿Cómo evoluciona algo a lo largo de una variable continua (normalmente el tiempo)? | Líneas | `ax.plot` |
| ¿Hay relación entre dos variables numéricas? | Dispersión | `ax.scatter` |
| ¿Cómo se comparan unas pocas categorías? | Barras | `ax.bar`, `ax.barh` |
| ¿Cómo se reparte una variable numérica? | Histograma | `ax.hist` |

### 4.1 Líneas

La línea afirma que **entre dos puntos consecutivos hay continuidad**. Por eso vale
para una temperatura a lo largo del mes y no vale para comparar cuatro países: unir
España con Francia con una línea no significa nada.

In [ ]:
# Serie temporal con media móvil, línea de referencia y el máximo anotado.
dias = np.arange(1, 31)
temperaturas = 20 + 5 * np.sin(dias / 5) + rng.normal(0, 1, 30)

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(dias, temperaturas,
        color="crimson",
        linewidth=2,
        marker="o",
        markersize=5,
        markerfacecolor="white",
        markeredgewidth=1.5,
        label="Temperatura diaria",
        alpha=0.9)

# Media móvil de 5 días. `mode="valid"` descarta los extremos donde la ventana no
# cabe entera, y por eso hay que recortar también el eje X: si no, los dos arrays
# tienen longitudes distintas y Matplotlib protesta.
ventana = 5
media_movil = np.convolve(temperaturas, np.ones(ventana) / ventana, mode="valid")
ax.plot(dias[ventana - 1:], media_movil, "b--", linewidth=2,
        label=f"Media móvil ({ventana} días)")

ax.axhline(y=20, color="gray", linestyle=":", linewidth=1,
           label="Media histórica")

ax.set_title("Evolución de la temperatura — enero", fontsize=13,
             fontweight="bold", pad=12)
ax.set_xlabel("Día del mes")
ax.set_ylabel("Temperatura (°C)")
ax.legend(loc="upper right", framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xlim(0, 31)

# Anotar el máximo: `xy` es el punto que se señala, `xytext` dónde va el texto.
i_max = int(np.argmax(temperaturas))
ax.annotate(f"Máximo\n{temperaturas[i_max]:.1f} °C",
            xy=(dias[i_max], temperaturas[i_max]),
            xytext=(dias[i_max] + 3, temperaturas[i_max] + 1.2),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.4),
            fontsize=9, ha="left")

fig.tight_layout()
plt.show()

print(f"Media  {temperaturas.mean():6.2f} °C")
print(f"Máxima {temperaturas.max():6.2f} °C (día {dias[i_max]})")
print(f"Mínima {temperaturas.min():6.2f} °C")

### 4.2 Dispersión

Un punto por observación, y las dos coordenadas son dos variables. Es el gráfico para
**buscar relación** entre variables, y admite dos dimensiones más: el color (`c`) y el
tamaño (`s`).

Cuidado con esas dos dimensiones extra. El color se lee bien; **el tamaño se lee
mal**, porque el ojo compara áreas y no radios, y casi todo el mundo lo interpreta al
revés. Si la variable que va en el tamaño importa, mejor otro panel.

In [ ]:
n = 200
horas_estudio = rng.uniform(0, 10, n)
calificacion = np.clip(50 + 4 * horas_estudio + rng.normal(0, 5, n), 0, 100)
asistencia = rng.uniform(60, 100, n)
trabajos = rng.integers(0, 10, n)

fig, ax = plt.subplots(figsize=(11, 6))

dispersion = ax.scatter(horas_estudio, calificacion,
                        c=asistencia,        # tercera variable: el color
                        s=trabajos * 20,     # cuarta variable: el tamaño
                        cmap="viridis",
                        alpha=0.7,
                        edgecolors="black",
                        linewidth=0.5)

# Recta de tendencia. `polyfit` de grado 1 es una regresión lineal por mínimos
# cuadrados; hay que ordenar la x antes de dibujar la recta o sale un garabato.
pendiente, interseccion = np.polyfit(horas_estudio, calificacion, 1)
x_orden = np.sort(horas_estudio)
ax.plot(x_orden, pendiente * x_orden + interseccion, "r--", linewidth=2,
        label=f"Tendencia: y = {pendiente:.2f}x + {interseccion:.2f}")

ax.set_title("Horas de estudio y calificación\n"
             "Color: asistencia · Tamaño: trabajos entregados",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Horas de estudio semanales")
ax.set_ylabel("Calificación final (%)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3, linestyle="--")

barra = fig.colorbar(dispersion, ax=ax)
barra.set_label("Asistencia (%)")

correlacion = float(np.corrcoef(horas_estudio, calificacion)[0, 1])
ax.text(0.03, 0.96, f"Correlación de Pearson: {correlacion:.3f}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.85))

fig.tight_layout()
plt.show()

print(f"Correlación horas-calificación: {correlacion:.3f}")
print()
print("Ojo: la correlación es alta porque los datos se han GENERADO con esa relación.")
print("Con datos reales, una correlación de 0,8 entre dos variables no dice que una")
print("cause la otra. Eso se trabaja en el cuaderno 06.")

### 4.3 Barras

Compara categorías. La longitud de la barra es lo que se lee, y por eso **el eje tiene
que empezar en cero**: si empieza en otro sitio, las longitudes ya no son
proporcionales a los valores y el gráfico miente. Es la manipulación más frecuente que
existe, y se ve en el cuaderno 06.

In [ ]:
lenguajes = ["Python", "JavaScript", "Java", "C++", "C#", "Go", "Rust", "TypeScript"]
uso_2023 = np.array([29.9, 19.6, 17.3, 12.5, 6.7, 2.8, 2.1, 1.9])
uso_2024 = np.array([32.1, 18.8, 16.1, 11.8, 7.2, 3.5, 2.8, 2.4])

posicion = np.arange(len(lenguajes))
ancho = 0.38

fig, ax = plt.subplots(figsize=(12, 6))

barras_2023 = ax.bar(posicion - ancho / 2, uso_2023, ancho, label="2023",
                     color="#7fb3d5", edgecolor="#1a5276", linewidth=1.2)
barras_2024 = ax.bar(posicion + ancho / 2, uso_2024, ancho, label="2024",
                     color="#f1948a", edgecolor="#922b21", linewidth=1.2)

ax.set_title("Uso declarado de lenguajes de programación", fontsize=13,
             fontweight="bold", pad=12)
ax.set_xlabel("Lenguaje")
ax.set_ylabel("Porcentaje de uso")
ax.set_xticks(posicion)
ax.set_xticklabels(lenguajes, rotation=45, ha="right")
ax.legend(title="Año")
ax.grid(True, axis="y", alpha=0.3, linestyle="--")
ax.set_ylim(0, 35)   # empieza en cero, que es la condición para que se pueda leer
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# `bar_label` pone la etiqueta encima de cada barra. Sustituye al bucle con
# `annotate` que había que escribir a mano hasta Matplotlib 3.4.
ax.bar_label(barras_2023, fmt="%.1f", fontsize=7, padding=2)
ax.bar_label(barras_2024, fmt="%.1f", fontsize=7, padding=2)

fig.tight_layout()
plt.show()

cambio = uso_2024 - uso_2023
print("Cambio de 2023 a 2024:")
for lenguaje, delta in sorted(zip(lenguajes, cambio), key=lambda p: -p[1]):
    flecha = "↑" if delta > 0 else "↓" if delta < 0 else "→"
    print(f"  {lenguaje:12s} {flecha} {delta:+.1f} puntos")

### 4.4 Histograma

Reparte los valores en intervalos y cuenta cuántos caen en cada uno. Es la forma de
ver **la forma de una distribución**: si es simétrica, si tiene cola, si tiene dos
picos.

El parámetro que decide todo es `bins`. Con pocos intervalos se aplana la forma y con
demasiados aparece ruido que no está en los datos. Conviene probar dos o tres valores
antes de quedarse con uno, y decirlo si el resultado depende de la elección.

In [ ]:
n = 1000
normal = rng.normal(170, 10, n)
uniforme = rng.uniform(150, 190, n)
sesgada = rng.gamma(5, 3, n) + 140

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Cuatro cosas que se ven en un histograma", fontsize=15,
             fontweight="bold")

# 1. Histograma básico, con la media y la mediana marcadas.
ax = axes[0, 0]
ax.hist(normal, bins=30, color="#7fb3d5", edgecolor="#1a5276", alpha=0.85)
ax.axvline(normal.mean(), color="red", linestyle="--", linewidth=2,
           label=f"media = {normal.mean():.1f}")
ax.axvline(np.median(normal), color="green", linestyle="--", linewidth=2,
           label=f"mediana = {np.median(normal):.1f}")
ax.set_title("Distribución normal (altura)", fontweight="bold")
ax.set_xlabel("Altura (cm)")
ax.set_ylabel("Frecuencia")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# 2. Normalizado a densidad, con la curva teórica encima.
#
# Con `density=True` el area total vale 1, y entonces el histograma es comparable con
# una funcion de densidad. La formula de la normal se escribe a mano a proposito: son
# dos lineas, y asi se ve que no hay magia detras.
ax = axes[0, 1]
ax.hist(normal, bins=30, density=True, color="#f1948a", edgecolor="#922b21",
        alpha=0.85, label="Datos")
mu, sigma = normal.mean(), normal.std()
x_teorico = np.linspace(normal.min(), normal.max(), 200)
densidad = np.exp(-0.5 * ((x_teorico - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
ax.plot(x_teorico, densidad, "b-", linewidth=2, label="Normal teórica")
ax.set_title("Densidad y curva teórica", fontweight="bold")
ax.set_xlabel("Altura (cm)")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# 3. Una distribución con cola a la derecha, y por qué la media deja de servir.
ax = axes[1, 0]
ax.hist(sesgada, bins=30, color="#82c39b", edgecolor="#1d6b3f", alpha=0.85)
ax.axvline(sesgada.mean(), color="red", linestyle="--", linewidth=2,
           label=f"media = {sesgada.mean():.1f}")
ax.axvline(np.median(sesgada), color="black", linestyle="--", linewidth=2,
           label=f"mediana = {np.median(sesgada):.1f}")
ax.set_title("Distribución con cola a la derecha", fontweight="bold")
ax.set_xlabel("Valor")
ax.set_ylabel("Frecuencia")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# 4. Tres distribuciones superpuestas. Con transparencia y con `histtype="step"`
# se pueden comparar; con barras opacas la última taparía a las otras dos.
ax = axes[1, 1]
for datos, etiqueta in ((normal, "Normal"), (uniforme, "Uniforme"),
                        (sesgada, "Sesgada")):
    ax.hist(datos, bins=30, histtype="step", linewidth=2, label=etiqueta)
ax.set_title("Tres distribuciones comparadas", fontweight="bold")
ax.set_xlabel("Valor")
ax.set_ylabel("Frecuencia")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

print("Distribución con cola a la derecha:")
print(f"  media   {sesgada.mean():.2f}")
print(f"  mediana {np.median(sesgada):.2f}")
print()
print("La media está por encima de la mediana porque la cola la arrastra. Con datos")
print("así, informar solo de la media da una idea equivocada del caso típico, y el")
print("histograma es lo que lo delata.")

### 4.5 El mismo `bins`, tres respuestas distintas

Merece una celda propia, porque es la trampa más silenciosa del histograma.

In [ ]:
# Una distribución con DOS picos, mirada con tres resoluciones.
bimodal = np.concatenate([rng.normal(160, 5, 500), rng.normal(182, 5, 500)])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, (5, 30, 200)):
    ax.hist(bimodal, bins=k, color="#7fb3d5", edgecolor="#1a5276")
    ax.set_title(f"bins={k}", fontweight="bold")
    ax.set_xlabel("Altura (cm)")
    ax.grid(True, alpha=0.3, axis="y")
axes[0].set_ylabel("Frecuencia")
fig.suptitle("Los mismos 1.000 datos: una montaña, dos picos, o ruido",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print("Con bins=5 la distribución parece de un solo pico: la estructura desaparece.")
print("Con bins=200 aparecen dientes que no están en los datos, solo en el muestreo.")
print("Con bins=30 se ven los dos grupos, que es lo que hay de verdad.")
print()
print("Conclusión práctica: bins es una DECISIÓN, y una decisión que cambia la")
print("conclusión hay que declararla. Si el hallazgo solo aparece con un valor")
print("concreto de bins, el hallazgo es del valor de bins y no de los datos.")

## 5. Varios paneles en una figura

### 5.1 Rejilla regular con `plt.subplots`

`plt.subplots(filas, columnas)` devuelve la figura y un array de `Axes`. Con una sola
fila o columna el array es de una dimensión; con más, de dos, y se indexa
`axes[fila, columna]`. Es la causa de la mitad de los `IndexError` al empezar.

Dos parámetros que ahorran trabajo: `sharex` y `sharey` obligan a que los paneles
usen los mismos límites, lo que es **imprescindible cuando se comparan**. Dos paneles
con escalas distintas puestos uno al lado del otro son un engaño casi seguro.

In [ ]:
x = np.linspace(0, 10, 300)
senal = np.sin(x) + 0.5 * np.sin(3 * x)
ruido = rng.normal(0, 0.3, len(x))
senal_ruidosa = senal + ruido

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("Análisis de una señal en cuatro paneles", fontsize=15,
             fontweight="bold")

axes[0, 0].plot(x, senal, color="#1a5276", linewidth=1.8)
axes[0, 0].set_title("Señal original", fontweight="bold")
axes[0, 0].set_xlabel("Tiempo (s)")
axes[0, 0].set_ylabel("Amplitud")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(x, senal_ruidosa, color="#922b21", linewidth=0.9, alpha=0.8,
                label="Con ruido")
axes[0, 1].plot(x, senal, color="#1a5276", linestyle="--", linewidth=1.8,
                alpha=0.7, label="Original")
axes[0, 1].set_title("Señal con ruido", fontweight="bold")
axes[0, 1].set_xlabel("Tiempo (s)")
axes[0, 1].set_ylabel("Amplitud")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(senal_ruidosa, bins=30, color="#82c39b", edgecolor="#1d6b3f",
                alpha=0.85)
axes[1, 0].axvline(senal_ruidosa.mean(), color="red", linestyle="--", linewidth=2,
                   label=f"media = {senal_ruidosa.mean():.3f}")
axes[1, 0].set_title("Reparto de amplitudes", fontweight="bold")
axes[1, 0].set_xlabel("Amplitud")
axes[1, 0].set_ylabel("Frecuencia")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3, axis="y")

# Espectro de frecuencias con la FFT de NumPy. `rfft` es la versión para señales
# reales: devuelve solo la mitad útil del espectro, que es lo que se quiere dibujar.
espectro = np.abs(np.fft.rfft(senal))
frecuencias = np.fft.rfftfreq(len(x), d=x[1] - x[0])
axes[1, 1].plot(frecuencias, 2 * espectro / len(x), color="#6c3483", linewidth=1.8)
axes[1, 1].set_title("Espectro de frecuencias (FFT)", fontweight="bold")
axes[1, 1].set_xlabel("Frecuencia (Hz)")
axes[1, 1].set_ylabel("Amplitud")
axes[1, 1].set_xlim(0, 1)
axes[1, 1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("La FFT encuentra los dos senos que compusieron la señal:")
picos = frecuencias[np.argsort(-espectro)[:2]]
print(f"  frecuencias dominantes: {picos[0]:.3f} y {picos[1]:.3f} Hz")
print(f"  esperadas: {1 / (2 * np.pi):.3f} y {3 / (2 * np.pi):.3f} Hz")

### 5.2 Rejillas irregulares con `GridSpec`

Cuando los paneles no son todos del mismo tamaño, `plt.subplots` no llega. `GridSpec`
define una rejilla y cada `Axes` ocupa el rectángulo de celdas que se le indique con
la sintaxis de los recortes de NumPy: `gs[0:2, :]` son las dos primeras filas
completas.

La regla de composición: **un panel grande con el mensaje principal y varios pequeños
con el detalle**. Nueve paneles del mismo tamaño no tienen jerarquía, y quien los mira
no sabe por dónde empezar.

In [ ]:
from matplotlib.gridspec import GridSpec

x = np.linspace(0, 20, 500)
y1 = np.sin(x)
y2 = np.cos(x)
y3 = np.sin(x) * np.exp(-x / 10)

fig = plt.figure(figsize=(13, 8))
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.3)

# El panel principal: las dos primeras filas, las tres columnas.
ax_principal = fig.add_subplot(gs[0:2, :])
ax_principal.plot(x, y1, label="sin(x)", linewidth=2)
ax_principal.plot(x, y2, label="cos(x)", linewidth=2)
ax_principal.plot(x, y3, label="sin(x)·e^(-x/10)", linewidth=2)
ax_principal.set_title("El mensaje principal va en el panel grande",
                       fontsize=13, fontweight="bold", pad=12)
ax_principal.set_xlabel("x")
ax_principal.set_ylabel("y")
ax_principal.legend(loc="upper right")
ax_principal.grid(True, alpha=0.3)

# Los tres paneles pequeños: la fila de abajo, una columna cada uno.
for columna, (datos, titulo) in enumerate([
        (y1, "Reparto de sin(x)"),
        (y2, "Reparto de cos(x)"),
        (None, "sin frente a cos")]):
    ax = fig.add_subplot(gs[2, columna])
    if datos is not None:
        ax.hist(datos, bins=20, color=f"C{columna}", alpha=0.8,
                edgecolor="black", linewidth=0.5)
        ax.set_ylabel("Frec.", fontsize=8)
    else:
        ax.scatter(y1, y2, c=x, cmap="viridis", s=6, alpha=0.6)
        ax.set_xlabel("sin(x)", fontsize=8)
        ax.set_ylabel("cos(x)", fontsize=8)
    ax.set_title(titulo, fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Composición con GridSpec: jerarquía entre paneles",
             fontsize=15, fontweight="bold")
plt.show()

## 6. Personalización

### 6.1 Texto: títulos, etiquetas, leyendas y anotaciones

Un gráfico sin etiquetas de eje no es un gráfico, es un adorno. La lista mínima:

1. **Título** que diga qué se está viendo, no cómo se llama la variable.
2. **Etiquetas de los dos ejes, con la unidad.** «Importe» no vale; «Importe (€)» sí.
3. **Leyenda** si hay más de una serie.
4. **Anotaciones** solo en lo que hay que mirar. Dos o tres como mucho: si todo está
   señalado, nada está señalado.

In [ ]:
x = np.linspace(0, 10, 200)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(x, np.sin(x), color="#1a5276", linewidth=2, label="Seno")
ax.plot(x, np.cos(x), color="#922b21", linestyle="--", linewidth=2, label="Coseno")
ax.plot(x, np.sin(x) * np.cos(x), color="#1d6b3f", linestyle=":", linewidth=2.5,
        label="Producto")

ax.set_title("Funciones trigonométricas\nEl subtítulo va en la segunda línea",
             fontsize=14, fontweight="bold", pad=16)
ax.set_xlabel("Ángulo (radianes)", fontsize=11, labelpad=8)
ax.set_ylabel("Amplitud", fontsize=11, labelpad=8)

leyenda = ax.legend(loc="upper right", fontsize=10, framealpha=0.9,
                    edgecolor="0.6", title="Funciones", title_fontsize=10)
leyenda.get_title().set_fontweight("bold")

# Una sola anotación, en lo único que hay que mirar.
ax.annotate("Primer máximo del seno",
            xy=(np.pi / 2, 1),
            xytext=(np.pi / 2 + 1.6, 0.55),
            arrowprops=dict(arrowstyle="->", color="#1a5276", lw=1.8,
                            connectionstyle="arc3,rad=0.25"),
            fontsize=10, color="#1a5276",
            bbox=dict(boxstyle="round", facecolor="#eaf2f8", alpha=0.9))

# `transform=ax.transAxes` cambia el sistema de coordenadas: (0,0) es la esquina
# inferior izquierda del Axes y (1,1) la superior derecha, independientemente de los
# valores de los datos. Es lo que hace falta para colocar texto en un sitio fijo.
ax.text(0.02, 0.03, "Coordenadas del Axes, no de los datos",
        transform=ax.transAxes, fontsize=8, color="0.4")

ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xlim(0, 10)
ax.set_ylim(-1.5, 1.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
plt.show()

### 6.2 Colores, estilos de línea y marcadores

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("Las formas de especificar el aspecto de una línea", fontsize=15,
             fontweight="bold")

x = np.linspace(0, 10, 30)

# Colores: cuatro notaciones equivalentes.
ax = axes[0, 0]
ax.plot(x, np.sin(x), "b-", linewidth=2, label="'b' — abreviatura")
ax.plot(x, np.sin(x) + 1, color="red", linewidth=2, label="color='red' — nombre")
ax.plot(x, np.sin(x) + 2, color="#FF6347", linewidth=2, label="color='#FF6347' — hex")
ax.plot(x, np.sin(x) + 3, color=(0.2, 0.7, 0.3), linewidth=2,
        label="color=(0.2, 0.7, 0.3) — RGB")
ax.set_title("Colores", fontweight="bold")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Estilos de línea.
ax = axes[0, 1]
for i, (estilo, nombre) in enumerate([("-", "sólida"), ("--", "discontinua"),
                                      ("-.", "punto y raya"), (":", "punteada")]):
    ax.plot(x, np.sin(x) + i, estilo, linewidth=2, label=f"'{estilo}' — {nombre}")
ax.set_title("Estilos de línea", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Marcadores.
ax = axes[1, 0]
marcadores = [("o", "círculo"), ("s", "cuadrado"), ("^", "triángulo"),
              ("D", "rombo"), ("*", "estrella")]
for i, (marca, nombre) in enumerate(marcadores):
    ax.plot(x[::3], np.sin(x[::3]) + i * 0.5,
            marker=marca, linestyle="-", markersize=8,
            markerfacecolor="white", markeredgecolor=f"C{i}",
            markeredgewidth=2, linewidth=1.2,
            label=f"'{marca}' — {nombre}")
ax.set_title("Marcadores", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Formato compacto: [color][marcador][línea] en una sola cadena.
ax = axes[1, 1]
for i, formato in enumerate(["bo-", "rs--", "g^:", "mD-."]):
    ax.plot(x, np.sin(x) + i, formato, linewidth=2, markersize=5,
            label=f"'{formato}'")
ax.set_title("Formato compacto [color][marcador][línea]", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("Abreviaturas de color: b azul, g verde, r rojo, c cian,")
print("                       m magenta, y amarillo, k negro, w blanco.")
print()
print("Y los colores del ciclo por defecto, que son los que conviene usar cuando no")
print("hay un motivo para elegir: C0, C1, C2... hasta C9.")

### 6.3 Hojas de estilo

En lugar de repetir la personalización en cada gráfico, Matplotlib tiene **hojas de
estilo**: colecciones de valores por defecto que se aplican de golpe. Se ven aquí y se
usan a fondo en el cuaderno 02, donde además se construye una propia.

La forma correcta de aplicarlas es `with plt.style.context(nombre):`, que las aplica
**solo dentro del bloque**. `plt.style.use(nombre)` cambia el estado global y afecta a
todo lo que venga después en el cuaderno, incluidas las celdas que ya habías
ejecutado y vuelvas a ejecutar. Eso hace que un cuaderno dé resultados distintos
según el orden en que se ejecuten las celdas, y es exactamente lo que hay que evitar.

In [ ]:
print(f"Matplotlib trae {len(plt.style.available)} hojas de estilo. Algunas:")
for nombre in sorted(plt.style.available)[:12]:
    print("  ", nombre)
print("   ...")

estilos = ["default", "ggplot", "bmh", "seaborn-v0_8-darkgrid"]
x = np.linspace(0, 10, 100)

fig = plt.figure(figsize=(13, 8))
fig.suptitle("La misma figura con cuatro hojas de estilo", fontsize=15,
             fontweight="bold")

for i, estilo in enumerate(estilos, start=1):
    with plt.style.context(estilo):
        ax = fig.add_subplot(2, 2, i)
        ax.plot(x, np.sin(x), linewidth=2, label="sin(x)")
        ax.plot(x, np.cos(x), linewidth=2, label="cos(x)")
        ax.plot(x, np.sin(x) * np.exp(-x / 10), linewidth=2, label="amortiguada")
        ax.set_title(f"'{estilo}'", fontweight="bold", fontsize=11)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 7. Guardar la figura

`fig.savefig(ruta)` escribe la figura a un fichero, y el formato sale de la extensión.
La decisión importante es **raster o vectorial**:

| Formato | Tipo | Para qué | Ojo |
|---|---|---|---|
| PNG | Raster | Web, presentaciones, Aules | Hay que fijar `dpi`; al ampliar se pixela |
| PDF | Vectorial | Informes, imprenta, memorias | Escala sin perder calidad y suele pesar menos |
| SVG | Vectorial | Web, y para retocar después en Inkscape | El texto sigue siendo texto y se puede buscar |
| JPG | Raster con pérdida | Casi nunca | Los bordes de texto y líneas salen sucios |

Dos parámetros que casi siempre hacen falta:

- `dpi=300` para PNG destinado a impresión o a proyector; 100 es demasiado poco.
- `bbox_inches="tight"` recorta el margen blanco de sobra. Sin él, una figura con las
  etiquetas del eje X rotadas sale cortada.

En lugar de creerse la tabla, vamos a medir cuánto pesa cada formato con la misma
figura.

In [ ]:
# Una figura de dos tipos: una con pocos elementos y otra con muchos puntos.
def figura_ligera():
    """Un gráfico de líneas normal: pocos elementos que dibujar."""
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.linspace(0, 10, 200)
    ax.plot(x, np.sin(x), linewidth=2, label="sin(x)")
    ax.plot(x, np.cos(x), "--", linewidth=2, label="cos(x)")
    ax.set_title("Figura con pocos elementos")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.grid(True, alpha=0.3)
    return fig


def figura_pesada():
    """Una nube de 50.000 puntos: 50.000 elementos que dibujar."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(rng.normal(size=50_000), rng.normal(size=50_000),
               s=3, alpha=0.2, edgecolors="none")
    ax.set_title("Figura con 50.000 puntos")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    return fig


def pesa(fig, formato, **opciones):
    """Guarda la figura en memoria y devuelve su tamaño en kilobytes.

    Se usa `io.BytesIO` en lugar de un fichero para no dejar basura en el disco:
    savefig acepta cualquier objeto que se pueda escribir.
    """
    memoria = io.BytesIO()
    fig.savefig(memoria, format=formato, bbox_inches="tight", **opciones)
    return len(memoria.getvalue()) / 1024


print(f"{'formato':>10} {'pocos elementos':>18} {'50.000 puntos':>16}")
print("-" * 46)
for formato, opciones in [("png", {"dpi": 100}), ("png", {"dpi": 300}),
                          ("pdf", {}), ("svg", {}), ("jpg", {"dpi": 300})]:
    ligera, pesada = figura_ligera(), figura_pesada()
    etiqueta = formato + (f" {opciones['dpi']}" if "dpi" in opciones else "")
    print(f"{etiqueta:>10} {pesa(ligera, formato, **opciones):>15.0f} KB "
          f"{pesa(pesada, formato, **opciones):>13.0f} KB")
    plt.close(ligera)
    plt.close(pesada)

print()
print("Lo que hay que leer en esta tabla:")
print()
print("1. Para una figura NORMAL, el PDF vectorial pesa menos que el PNG a 300 puntos")
print("   por pulgada y encima no se pixela. Es la opción por defecto para un informe.")
print("2. Para una figura con MUCHOS PUNTOS se invierte: el vectorial tiene que guardar")
print("   cada uno de los 50.000 puntos, y se dispara. El raster pesa lo mismo tenga")
print("   dos puntos o dos millones, porque solo guarda píxeles.")
print("3. El JPG no ahorra nada aquí y estropea los bordes: en gráficos no se usa.")
print()
print("El arreglo para el caso 2 se llama `rasterized=True` y está en el cuaderno 02.")

## 8. El mismo dato, dos gráficos

Todo lo anterior es sintaxis. Esta sección es el criterio, y es lo que separa un
gráfico de un adorno. Los mismos seis meses de ventas y costes, dibujados dos veces.

In [ ]:
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun"]
ventas = np.array([45, 52, 48, 61, 58, 67])
costes = np.array([30, 33, 31, 38, 36, 40])

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 5.5))

# ANTES.
izq.plot(meses, ventas, color="red")
izq.plot(meses, costes, color="blue")
izq.set_title("Datos")
izq.set_ylim(28, 70)          # escala truncada: exagera las diferencias
izq.text(0.5, 0.94, "MAL", transform=izq.transAxes, fontsize=13,
         fontweight="bold", color="#922b21", ha="center", va="top",
         bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))

# DESPUÉS.
der.plot(meses, ventas, color="#2E86AB", linewidth=2.5, marker="o", markersize=8,
         markerfacecolor="white", markeredgewidth=2, label="Ventas")
der.plot(meses, costes, color="#A23B72", linewidth=2.5, marker="s", markersize=8,
         markerfacecolor="white", markeredgewidth=2, label="Costes")
der.fill_between(range(len(meses)), costes, ventas, alpha=0.15, color="#1d6b3f",
                 label="Margen")
der.set_title("Ventas y costes, primer semestre\nEl margen crece un 39 % en seis meses",
              fontsize=12, fontweight="bold", pad=12)
der.set_xlabel("Mes")
der.set_ylabel("Miles de euros")
der.legend(loc="upper left", framealpha=0.9)
der.grid(True, alpha=0.3, linestyle="--")
der.set_ylim(0, 75)           # empieza en cero
der.spines["top"].set_visible(False)
der.spines["right"].set_visible(False)
for i, (v, c) in enumerate(zip(ventas, costes)):
    if i == len(ventas) - 1:
        der.annotate(f"{v} k€", xy=(i, v), xytext=(6, 4),
                     textcoords="offset points", fontsize=9,
                     fontweight="bold", color="#2E86AB")
        der.annotate(f"{c} k€", xy=(i, c), xytext=(6, -14),
                     textcoords="offset points", fontsize=9,
                     fontweight="bold", color="#A23B72")
der.text(0.5, 0.94, "BIEN", transform=der.transAxes, fontsize=13,
         fontweight="bold", color="#1d6b3f", ha="center", va="top",
         bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))

fig.tight_layout()
plt.show()

margen_inicial = ventas[0] - costes[0]
margen_final = ventas[-1] - costes[-1]
print(f"Margen en enero: {margen_inicial} k€ · en junio: {margen_final} k€ "
      f"({(margen_final / margen_inicial - 1) * 100:+.0f} %)")
print()
print("Diferencias, una a una:")
print("  - El título del segundo dice LA CONCLUSIÓN, no el nombre del fichero.")
print("  - Los ejes están etiquetados y con unidad.")
print("  - Hay leyenda: en el primero no se sabe qué línea es cuál.")
print("  - El eje Y empieza en cero. En el primero empieza en 28, y eso hace que")
print("    junio parezca el triple que enero cuando es un 49 % más.")
print("  - El área sombreada muestra el margen, que es de lo que se quería hablar.")
print("  - Se han quitado los bordes de arriba y de la derecha: no informan de nada.")
print()
print("Ninguna de esas seis cosas es decoración. Las seis cambian lo que se entiende.")

## Ejercicios

### Instrucciones

Los ejercicios van de menos a más y todos se resuelven con lo que hay en este
cuaderno. Tres condiciones que se aplican a todos:

1. **Interfaz orientada a objetos.** Nada de `plt.plot`: `fig, ax = plt.subplots()` y
   después métodos de `ax`.
2. **Título, etiquetas de los dos ejes con su unidad, y leyenda** si hay más de una
   serie. Sin eso, el ejercicio no está hecho.
3. **Semilla fija** en todo lo que tenga aleatoriedad.

### Ejercicio 1 (Básico): El primer gráfico

Crea un array `x` con 100 puntos entre 0 y 10 y dibuja `y = x²` con la interfaz
orientada a objetos. Título «Función cuadrática», ejes etiquetados, rejilla, línea
roja continua.

In [ ]:
# TODO: Escribe tu código aquí
# Pista: np.linspace(0, 10, 100) y fig, ax = plt.subplots()

### Ejercicio 2 (Básico): Tres series y una leyenda

Dibuja `sin(x)`, `cos(x)` y `sin(2x)` entre 0 y 2π en el mismo `Axes`, cada una con un
color y un estilo de línea distintos, y con la leyenda puesta.

Después responde en una celda de texto: **¿por qué en este caso el eje Y no tiene que
empezar en cero?** Compáralo con la norma del gráfico de barras.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 3 (Básico): Dispersión y tendencia

Genera 100 puntos con `x` uniforme entre 0 y 10 e `y = 2x + 5 + ruido`, con ruido
normal de desviación 2. Dibuja la nube con `alpha=0.6`, añade la recta de tendencia
con `np.polyfit` y escribe la correlación dentro del gráfico con
`transform=ax.transAxes`.

In [ ]:
# TODO: Escribe tu código aquí
# Pista: la correlación es np.corrcoef(x, y)[0, 1]

### Ejercicio 4 (Intermedio): Barras agrupadas

Con los datos de la celda siguiente, dibuja las ventas de los tres productos por
trimestre en barras agrupadas: tres barras por trimestre, colores distintos, leyenda,
rejilla solo en el eje Y, y el valor encima de cada barra con `ax.bar_label`.

El eje Y tiene que empezar en cero. Explica en una línea por qué aquí sí es
obligatorio.

In [ ]:
trimestres = ["Q1", "Q2", "Q3", "Q4"]
producto_a = np.array([23, 28, 31, 35])
producto_b = np.array([18, 22, 27, 30])
producto_c = np.array([15, 19, 22, 28])

# TODO: Escribe tu código aquí
# Pista: con tres series el desplazamiento es -ancho, 0, +ancho

### Ejercicio 5 (Intermedio): Un panel de cuatro

Genera datos climáticos inventados de un año (temperatura mensual, humedad y
precipitación) y monta una figura 2×2 con: la temperatura mensual en líneas, su
histograma, la dispersión de temperatura frente a humedad, y la precipitación por
estación en barras.

Los cuatro paneles con título y ejes etiquetados, y la figura con `suptitle`.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 6 (Intermedio): Una función que dibuja

Escribe `dibuja_histograma(ax, datos, titulo, bins=30)` siguiendo la norma de la
sección 3.3: recibe el `Axes`, dibuja el histograma con la media y la mediana
marcadas, **no** llama a `plt.show()` y **no** imprime nada.

Demuestra que funciona usándola dos veces: en una figura de un panel y en una de
1×3 con tres distribuciones distintas.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 7 (Intermedio): El histograma y su `bins`

Con los datos bimodales de la celda siguiente, encuentra **el valor más pequeño de
`bins` con el que se distinguen los dos picos**. Dibuja tres histogramas: uno por
debajo de ese valor, ese valor, y uno muy por encima.

Responde: si tuvieras que informar de una sola cifra sobre estos datos, ¿la media?
¿Por qué no?

In [ ]:
rng_ej = np.random.default_rng(7)
datos_bimodales = np.concatenate([rng_ej.normal(20, 3, 400),
                                  rng_ej.normal(38, 4, 300)])

# TODO: Escribe tu código aquí

### Ejercicio 8 (Avanzado): Composición con GridSpec

Monta esta composición con `GridSpec`:

```
┌─────────────────────────────┬─────────┐
│                             │         │
│        PRINCIPAL            │  LATERAL│
│                             │         │
├─────────┬─────────┬─────────┤         │
│  SUB 1  │  SUB 2  │  SUB 3  │         │
└─────────┴─────────┴─────────┴─────────┘
```

El panel lateral no lleva datos: lleva texto con un resumen numérico, y para eso hay
que quitarle los ejes con `ax.axis("off")`. Usa los datos que quieras, pero que los
cinco paneles hablen del mismo conjunto.

In [ ]:
# TODO: Escribe tu código aquí
# Pista: GridSpec(3, 4); el principal es gs[0:2, 0:3] y el lateral gs[:, 3]

### Ejercicio 9 (Avanzado): Una figura para imprimir

Con los datos de la celda siguiente —exactitud de tres modelos según el tamaño del
conjunto de entrenamiento— produce **una figura para un informe impreso**:

1. Tamaño exacto 6,5 × 4 pulgadas, que es el ancho de una columna de revista.
2. Tipografía con serifas y cuerpos pequeños (8 a 10 puntos).
3. Sin colores llamativos: gris y un solo color de acento.
4. Leyenda dentro del área de dibujo.
5. Eje X en escala logarítmica, que es la que corresponde a estos tamaños.
6. Exportada a PDF **y** a PNG de 300 puntos por pulgada, midiendo lo que pesa cada
   una con la función `pesa` de la sección 7.

Y responde: para este informe, ¿cuál de las dos entregarías, y por qué?

In [ ]:
tamanos = np.array([100, 500, 1000, 5000, 10_000, 50_000])
exactitud_a = np.array([0.65, 0.72, 0.78, 0.85, 0.88, 0.90])
exactitud_b = np.array([0.68, 0.75, 0.81, 0.87, 0.90, 0.92])
exactitud_c = np.array([0.62, 0.70, 0.76, 0.83, 0.87, 0.91])

# TODO: Escribe tu código aquí
# Pista: usa `with plt.style.context({...}):` con un diccionario de parámetros,
# y ax.set_xscale("log")

### Ejercicio 10 (Avanzado): Deshacer un gráfico malo

La celda siguiente dibuja las ventas de TechStore por categoría con **cinco defectos
graves de golpe**. Tu trabajo:

1. Ejecútala y **enumera los cinco defectos** en una celda de texto, uno por línea.
2. Vuelve a dibujar los mismos datos arreglándolos todos.
3. Escribe el título del gráfico nuevo de forma que diga **la conclusión**, no el
   nombre de las variables.

In [ ]:
categorias = ["Informática", "Audio", "Periféricos", "Telefonía",
              "Almacenamiento", "Vestible"]
importe = np.array([182_400, 91_200, 74_500, 68_900, 31_200, 12_800])

# El gráfico con los cinco defectos. No lo copies: arréglalo.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(categorias)), importe, color=["red", "lime", "blue",
                                               "yellow", "magenta", "cyan"])
ax.set_ylim(10_000, 190_000)
ax.set_xticks(range(len(categorias)))
ax.set_xticklabels(categorias)
ax.set_title("importe")
plt.show()

# TODO: Los cinco defectos, y después el gráfico arreglado

## Resumen

1. **`Figure` es el lienzo, `Axes` la zona de dibujo, `Axis` un eje.** Casi todo lo
   que se escribe son métodos de un `Axes`.
2. **Interfaz orientada a objetos siempre.** `plt.` mantiene estado global y eso hace
   que una función que dibuja no se pueda escribir bien.
3. **Una función que dibuja recibe el `Axes` y no muestra ni guarda nada.** Es lo que
   la hace reutilizable.
4. **Cada gráfico responde a un tipo de pregunta.** La línea afirma continuidad; la
   barra compara categorías y necesita el cero; el histograma muestra la forma de una
   distribución; la dispersión busca relación.
5. **`bins` es una decisión que cambia la conclusión.** Si el hallazgo depende de
   `bins`, el hallazgo no es de los datos.
6. **Al comparar paneles, `sharex` y `sharey`.** Dos escalas distintas lado a lado
   engañan casi siempre.
7. **Vectorial para figuras normales, raster para nubes de muchos puntos.** Y está
   medido, no supuesto.
8. **Título con la conclusión, ejes con unidad, leyenda si hay varias series.** Lo
   demás es opcional; esto no.

## Para seguir

- [Guía de usuario de Matplotlib](https://matplotlib.org/stable/users/index.html)
- [Galería de ejemplos](https://matplotlib.org/stable/gallery/index.html) — la forma
  práctica de usarla es buscar el gráfico que te hace falta y copiar el código.
- [Hojas de referencia oficiales](https://matplotlib.org/cheatsheets/) — tres páginas
  que merece la pena tener a mano.
- Jake VanderPlas, *Python Data Science Handbook*, capítulo 4. Disponible en abierto.

**Siguiente:** el cuaderno 02 va a la personalización a fondo, a los mapas de color y
al rendimiento cuando hay muchos datos.